# GeoLife CP1 — Final Stay-point Sensitivity Validation

Notebook 02 đã chứng minh production pipeline chạy được trên full release. Notebook 02b trả lời câu hỏi tiếp theo:

> **Frozen baseline có nằm trong một vùng parameter behavior hợp lý hay chỉ tình cờ tốt trên một cấu hình duy nhất?**

### Mục tiêu

1. reconcile full-release baseline cache với complete audit-event semantics;
2. kiểm tra hard-speed pathology có tập trung hay lan rộng;
3. tránh prefix-sampling bias bằng deterministic user-stratified sample;
4. chạy đủ **27 configs**;
5. thêm user-level repeated-location proxy trước khi freeze CP1.

> Notebook này **không tối ưu Home/Office accuracy**. GeoLife không có direct Home/Office ground truth. Sensitivity được dùng để tránh chọn một threshold nằm ở unstable/extreme edge.

### Quan hệ với notebook khác

- notebook 02: implementation + full-release baseline;
- notebook 02b: final parameter sensitivity;
- notebook 02c: mentor follow-up audit về semantics của same-second >10 m.

02c về sau đổi cách gọi `>10m` từ “conflict/corruption” sang **unresolved spatial ambiguity**, nhưng boundary behavior không đổi nên kết quả sensitivity ở đây vẫn hợp lệ.

In [1]:

from pathlib import Path
from time import perf_counter
from IPython.display import display
import os, subprocess, sys
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
CACHE_DIR = Path(os.environ.get("GEOLIFE_CACHE_DIR", "/mnt/geolife-data/cache/cp1_staypoints"))

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        Path("/mnt/geolife-data/extracted/Geolife Trajectories 1.3/Data"),
        Path("/mnt/geolife-data/Data"),
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(Path("/mnt/geolife-data").glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
sys.path.insert(0, str(REPO_DIR / "src"))

from geolife.geo.distance import haversine_m
from geolife.staypoints import clean_trajectory, clean_trajectory_with_audit, detect_staypoints

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

def read_plt(path):
    df = pd.read_csv(
        path, skiprows=6, header=None,
        names=["latitude","longitude","unused","altitude","date_days","date","time"],
    )
    df["timestamp"] = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"].astype(str),
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
        utc=True,
    )
    if df["timestamp"].isna().any():
        raise ValueError(f"Unparsable timestamp in {path}")
    return df[["timestamp","latitude","longitude"]]

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))
print("DATA_ROOT:", DATA_ROOT)
print("Trajectory files:", len(files))


Cloning into '/tmp/geolife'...


DATA_ROOT: /mnt/geolife-data/extracted/Geolife Trajectories 1.3/Data
Trajectory files: 18670


## 1. Vì sao cần thêm audit log ngoài bảng cleaned?

Ở notebook 02, sau khi cleaning ta có một bảng `cleaned`.

Trong bảng này, mỗi point được giữ lại có thể có cột:

```text
boundary_before_reason
```

Cột này cho biết:

> Trước point này đã có một sự kiện làm trajectory bị ngắt thành sequence mới.

Ví dụ:

```text
A retained
B bị loại vì same-second ambiguity
C retained
```

thì ở point `C` ta có thể ghi:

```text
boundary_before_reason = same_second_spatial_ambiguity
```

Nhờ vậy ta biết rằng giữa `A` và `C` đã xảy ra một boundary.

### Vấn đề là gì?

Không phải cleaning event nào cũng có thể gắn vào một retained row.

Ví dụ nếu event xảy ra ở cuối trajectory:

```text
A retained
B bị loại vì ambiguity
END
```

thì không còn point nào phía sau để ghi `boundary_before_reason`.

Ngoài ra, nếu nhiều events xảy ra trước cùng một retained point thì một ô `boundary_before_reason` cũng không thể lưu đầy đủ tất cả events đó.

Vì vậy ta dùng hai thứ cho hai mục đích khác nhau:

```text
boundary_before_reason
→ giải thích vì sao cleaned trajectory bị tách sequence

clean_trajectory_with_audit()
→ ghi đầy đủ từng cleaning event đã xảy ra
```

Nói ngắn gọn:

> `cleaned` cho biết dữ liệu cuối cùng trông như thế nào, còn `audit` cho biết trong quá trình cleaning đã xảy ra những gì.

Probe bên dưới cố ý tạo một ambiguity ở cuối trajectory để kiểm tra rằng event này vẫn xuất hiện trong audit log, dù không thể gắn vào `boundary_before_reason`.


In [2]:

BASELINE_CACHE = CACHE_DIR / "baseline_summary_v2.pkl"
if not BASELINE_CACHE.exists():
    raise FileNotFoundError(
        f"{BASELINE_CACHE} not found. Run notebook 02 full baseline first."
    )

baseline_summary = pd.read_pickle(BASELINE_CACHE)
if len(baseline_summary) != len(files):
    raise RuntimeError(
        f"Baseline cache rows={len(baseline_summary):,}, expected={len(files):,}"
    )

print("Total stays:", int(baseline_summary["n_stays"].sum()))
print("Files with >=1 stay:", int((baseline_summary["n_stays"] > 0).sum()))
print("Attached conflict reasons:", int(baseline_summary["same_second_conflict_boundaries"].sum()))
print("EDA exact reference: same-second >10m ambiguity groups=835, invalid points=1")

display(
    baseline_summary.nlargest(10, "hard_speed_boundaries")[
        ["file","raw_rows","clean_rows","n_sequences","n_stays",
         "temporal_gap_boundaries","hard_speed_boundaries"]
    ]
)

# Terminal conflict: no later retained row exists, but event must remain observable.
probe = pd.DataFrame(
    [
        ("2026-01-01T09:59:00Z", 39.0, 116.0),
        ("2026-01-01T10:00:00Z", 39.0, 116.0),
        ("2026-01-01T10:00:00Z", 39.02, 116.0),
    ],
    columns=["timestamp","latitude","longitude"],
)
probe["timestamp"] = pd.to_datetime(probe["timestamp"], utc=True)
cleaned_probe, audit_probe = clean_trajectory_with_audit(probe)
display(audit_probe)
assert (audit_probe["reason"] == "same_second_spatial_ambiguity").sum() == 1
print("Complete audit-event smoke check: OK")


Total stays: 5821
Files with >=1 stay: 2435
Attached conflict reasons: 683
EDA exact reference: same-second >10m ambiguity groups=835, invalid points=1


,file,raw_rows,clean_rows,n_sequences,n_stays,temporal_gap_boundaries,hard_speed_boundaries
7030,/mnt/geolife-data/extracted/Geolife Trajectori...,8117,8065,623,0,3,593
8668,/mnt/geolife-data/extracted/Geolife Trajectori...,2444,2444,68,2,23,44
1403,/mnt/geolife-data/extracted/Geolife Trajectori...,4480,4441,21,0,3,16
14661,/mnt/geolife-data/extracted/Geolife Trajectori...,41728,41728,74,17,59,14
1417,/mnt/geolife-data/extracted/Geolife Trajectori...,15314,14875,39,0,9,10
3927,/mnt/geolife-data/extracted/Geolife Trajectori...,7102,7094,24,0,12,7
18574,/mnt/geolife-data/extracted/Geolife Trajectori...,50,50,11,0,4,6
7052,/mnt/geolife-data/extracted/Geolife Trajectori...,4296,4246,33,0,18,5
14256,/mnt/geolife-data/extracted/Geolife Trajectori...,161,161,15,0,9,5
6107,/mnt/geolife-data/extracted/Geolife Trajectori...,118,118,5,0,1,3


,timestamp,reason
0,2026-01-01 10:00:00+00:00,same_second_spatial_ambiguity


Complete audit-event smoke check: OK


### Probe bên dưới đang kiểm tra gì?

Probe cố ý tạo một ambiguity ở **cuối trajectory**:

```text
09:59  → point bình thường

10:00  → point A: (39.00, 116.0)
10:00  → point B: (39.02, 116.0)
```

Hai point cuối có:

```text
cùng timestamp = 10:00
nhưng vị trí cách nhau rất xa
```

nên cleaning phải tạo event:

```text
same_second_spatial_ambiguity
```

Quan trọng là sau nhóm `10:00` này **không còn point nào nữa**.

Do đó event này không thể được attach vào một retained row phía sau bằng `boundary_before_reason`.

Nhưng khi dùng:

```python
cleaned_probe, audit_probe = clean_trajectory_with_audit(probe)
```

audit log vẫn phải ghi lại event đó.

Kết quả:

```text
timestamp                  reason
2026-01-01 10:00:00+00:00 same_second_spatial_ambiguity
```

nghĩa là:

> dù ambiguity nằm ở cuối trajectory và không có retained row phía sau, event vẫn không bị mất khỏi audit log.

Assertion:

```python
assert (
    audit_probe["reason"] == "same_second_spatial_ambiguity"
).sum() == 1
```

kiểm tra rằng đúng **1 ambiguity event** đã được ghi nhận.

Đây chính là lý do ta cần `clean_trajectory_with_audit()` bên cạnh `boundary_before_reason`.


### Cách đọc output phần 1

Expected full-release baseline:

- **5,821 stays**;
- **2,435 files** có >=1 stay;
- exact EDA reference có **835 same-second >10 m ambiguity groups**;
- chỉ **1 invalid coordinate** trong release.

Attached boundary counts trong `baseline_summary_v2.pkl` có thể thấp hơn exact event count, và đó **không phải bug** nếu discrepancy đến từ terminal/multiple discarded events.

Hard-speed outliers cũng tập trung mạnh: trajectory pathology lớn nhất có **593 hard-speed boundaries**. Điều này ủng hộ strategy cắt continuity ở extreme segments thay vì áp một transport-speed filter toàn cục.

## 2. Chọn sample cân bằng hơn giữa các user

### Vì sao không lấy `files[:N]`?

Số lượng trajectory file của mỗi user trong GeoLife rất khác nhau.

Nếu chỉ lấy:

```python
files[:N] (N file đầu tiên)
```

thì sample phụ thuộc vào thứ tự file trên disk. Một vài user có nhiều file ở đầu danh sách có thể chiếm phần lớn sample.

Khi đó sensitivity analysis dễ bị lệch theo các user này thay vì phản ánh toàn bộ dataset.

### Cách chọn sample

Ta dùng rule sau:

* cố gắng bao phủ tất cả user có trajectory;
* mỗi user lấy tối đa 5 files;
* nếu user có hơn 5 files, không lấy 5 file đầu mà chọn các file trải đều theo toàn bộ lịch sử của user.

Ví dụ một user có 100 files thì có thể lấy gần các vị trí:

```text
file 1
file 25
file 50
file 75
file 100
```

thay vì:

```text
file 1
file 2
file 3
file 4
file 5
```

Cách này giúp sample đại diện tốt hơn cho nhiều giai đoạn trong lịch sử của mỗi user.

### Sample cuối cùng

```text
182 users
851 trajectory files
tối đa 5 files / user
```

Đây không phải random sample theo nghĩa thống kê.

Nó là một **sample được chọn theo rule cố định và cân bằng theo user**, nhằm tránh việc những user có quá nhiều trajectory files chi phối kết quả sensitivity analysis.


In [3]:

MAX_FILES_PER_USER = 5

def user_id_from_path(path):
    return path.parent.parent.name

def build_user_stratified_sample(paths, max_files_per_user=5):
    by_user = {}
    for path in paths:
        by_user.setdefault(user_id_from_path(path), []).append(path)

    sample = []
    for user_id in sorted(by_user):
        user_paths = sorted(by_user[user_id])
        k = min(max_files_per_user, len(user_paths))
        if k == len(user_paths):
            selected = user_paths
        else:
            idx = np.unique(np.linspace(0, len(user_paths) - 1, num=k, dtype=int))
            selected = [user_paths[int(i)] for i in idx]
        sample.extend((user_id, path) for path in selected)
    return sample

sample = build_user_stratified_sample(files, MAX_FILES_PER_USER)
manifest = pd.DataFrame([{"user_id": u, "file": str(p)} for u,p in sample])

print("Users covered:", manifest["user_id"].nunique())
print("Files sampled:", len(manifest))
display(manifest.groupby("user_id").size().rename("sampled_files").describe())
assert manifest["user_id"].nunique() == len({user_id_from_path(p) for p in files})


Users covered: 182
Files sampled: 851


count    182.000000
mean       4.675824
std        0.951514
min        1.000000
25%        5.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: sampled_files, dtype: float64

### Cách đọc sample manifest

`sampled_files` cho biết mỗi user đóng góp bao nhiêu trajectory vào sensitivity.

Điều quan trọng là coverage theo **user**, không phải point count. Một user có 2,000 trajectories và một user có 2 trajectories không còn tạo chênh lệch hàng nghìn lần trong sensitivity surface.

**Không được suy ra:** 851 files là representative sample theo xác suất của toàn bộ movement population. Mục tiêu ở đây là robustness của engineering thresholds dưới user-level balancing.

## 3. So sánh 27 cấu hình stay-point và độ lặp lại theo user

Ta thử nhiều bộ tham số khác nhau để xem kết quả stay-point thay đổi như thế nào.

Các giá trị được thử:

```text
max_gap_s            = 120 / 300 / 600
distance_threshold_m = 100 / 200 / 300
min_dwell_s          = 600 / 1200 / 1800
```

Có 3 lựa chọn cho mỗi tham số, nên tổng cộng:

```text
3 × 3 × 3 = 27 cấu hình
```

### Ta đo những gì?

Với mỗi cấu hình, ta ghi lại:

* `n_stays`: tổng số stay được phát hiện;
* `users_with_stays`: số user có ít nhất một stay;
* `median_stays_per_active_user`: số stay điển hình của một user có stay;
* `median_duration_s`, `p90_duration_s`: thời lượng stay điển hình và phần đuôi dài;
* `users_with_2plus_stays`: số user có ít nhất 2 stays;
* `repeat_location_users`: số user có ít nhất 2 stays xuất hiện gần cùng một vị trí;
* `repeat_location_user_rate`: tỷ lệ user có vị trí lặp lại trong nhóm user có từ 2 stays trở lên.

### Vì sao cần kiểm tra vị trí lặp lại?

Home và Office thường không phải là nơi user chỉ xuất hiện một lần.

Nếu một cấu hình tạo ra rất nhiều stays nhưng phần lớn đều là các vị trí chỉ xuất hiện một lần, thì số stay cao chưa chắc đã hữu ích cho bước Home/Office sau này.

Vì vậy ta dùng một kiểm tra đơn giản:

```text
hai stay representatives cách nhau <= 200 m
→ xem như có dấu hiệu quay lại cùng một vùng
```

Sau đó tính:

```text
repeat_location_user_rate
=
repeat_location_users
/
users_with_2plus_stays
```

Metric này giúp trả lời câu hỏi:

> Trong số những user có đủ ít nhất 2 stays để so sánh, có bao nhiêu user thực sự quay lại một vùng gần giống nhau?

### Lưu ý

Đây chỉ là **recurrence proxy**, không phải độ chính xác Home/Office.

Hai stays gần nhau có thể là:

* nhà;
* văn phòng;
* quán cà phê;
* ga tàu;
* hoặc một địa điểm thường xuyên khác.

Ta chỉ dùng metric này để xem cấu hình nào tạo ra các stay patterns có tính lặp lại hợp lý hơn cho bước semantic inference phía sau.


In [4]:

GAPS = [120, 300, 600]
DISTANCES = [100, 200, 300]
DWELLS = [600, 1200, 1800]
REPEAT_RADIUS_M = 200.0
SENSITIVITY_CACHE = CACHE_DIR / "sensitivity_user_stratified_v1.pkl"

grid = [(g,d,w) for g in GAPS for d in DISTANCES for w in DWELLS]

def repeat_metrics(stays_by_user):
    users_2plus = repeat_users = 0
    for coords_list in stays_by_user.values():
        if len(coords_list) < 2:
            continue
        users_2plus += 1
        coords = np.asarray(coords_list, dtype=float)
        hit = False
        for i in range(len(coords) - 1):
            distances = np.asarray(
                haversine_m(
                    coords[i,0], coords[i,1],
                    coords[i+1:,0], coords[i+1:,1],
                ),
                dtype=float,
            )
            if np.any(distances <= REPEAT_RADIUS_M):
                hit = True
                break
        repeat_users += int(hit)
    return users_2plus, repeat_users, (
        repeat_users / users_2plus if users_2plus else np.nan
    )

def evaluate_grid(sample):
    accum = {
        key: {"n_stays":0, "files_with_stays":0, "durations":[], "by_user":{}}
        for key in grid
    }
    configs_by_gap = {
        gap: [(d,w) for g,d,w in grid if g == gap]
        for gap in GAPS
    }

    t0 = perf_counter()
    for i, (user_id, path) in enumerate(sample, 1):
        raw = read_plt(path)
        for gap in GAPS:
            cleaned = clean_trajectory(
                raw,
                same_second_radius_m=BASELINE["same_second_radius_m"],
                max_gap_s=gap,
                hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
            )
            for distance_m, dwell_s in configs_by_gap[gap]:
                bucket = accum[(gap,distance_m,dwell_s)]
                stays = detect_staypoints(
                    cleaned,
                    distance_threshold_m=distance_m,
                    min_dwell_s=dwell_s,
                )
                if stays.empty:
                    continue
                bucket["n_stays"] += len(stays)
                bucket["files_with_stays"] += 1
                bucket["durations"].extend(stays["duration_s"].astype(float).tolist())
                bucket["by_user"].setdefault(user_id, []).extend(
                    stays[["latitude","longitude"]].to_numpy(dtype=float).tolist()
                )
        if i % 100 == 0:
            print(f"{i:,}/{len(sample):,} files | {(perf_counter()-t0)/60:.1f} min")

    users = sorted({u for u,_ in sample})
    rows = []
    for gap,distance_m,dwell_s in grid:
        bucket = accum[(gap,distance_m,dwell_s)]
        counts = np.array([len(bucket["by_user"].get(u, [])) for u in users], dtype=float)
        active = counts[counts > 0]
        durations = bucket["durations"]
        users_2plus, repeat_users, repeat_rate = repeat_metrics(bucket["by_user"])
        rows.append({
            "max_gap_s": gap,
            "distance_threshold_m": distance_m,
            "min_dwell_s": dwell_s,
            "n_stays": bucket["n_stays"],
            "files_with_stays": bucket["files_with_stays"],
            "users_with_stays": int((counts > 0).sum()),
            "mean_stays_per_user": float(counts.mean()),
            "median_stays_per_active_user": float(np.median(active)) if active.size else np.nan,
            "users_with_2plus_stays": users_2plus,
            "repeat_location_users": repeat_users,
            "repeat_location_user_rate": repeat_rate,
            "median_duration_s": float(np.median(durations)) if durations else np.nan,
            "p90_duration_s": float(np.quantile(durations, 0.9)) if durations else np.nan,
        })
    return pd.DataFrame(rows)

if SENSITIVITY_CACHE.exists():
    sensitivity = pd.read_pickle(SENSITIVITY_CACHE)
    print("Loaded:", SENSITIVITY_CACHE)
else:
    sensitivity = evaluate_grid(sample)
    sensitivity.to_pickle(SENSITIVITY_CACHE)
    print("Saved:", SENSITIVITY_CACHE)

display(sensitivity.sort_values(["max_gap_s","distance_threshold_m","min_dwell_s"]))

baseline_row = sensitivity[
    (sensitivity["max_gap_s"] == 300)
    & (sensitivity["distance_threshold_m"] == 200)
    & (sensitivity["min_dwell_s"] == 1200)
]
print("Baseline config:")
display(baseline_row)


Loaded: /mnt/geolife-data/cache/cp1_staypoints/sensitivity_user_stratified_v1.pkl


,max_gap_s,distance_threshold_m,min_dwell_s,n_stays,files_with_stays,users_with_stays,mean_stays_per_user,median_stays_per_active_user,users_with_2plus_stays,repeat_location_users,repeat_location_user_rate,median_duration_s,p90_duration_s
0,120,100,600,592,174,90,3.252747,4.0,68,53,0.779412,882.5,1858.7
1,120,100,1200,146,66,42,0.802198,2.0,27,22,0.814815,1704.0,3732.5
2,120,100,1800,69,41,31,0.379121,1.0,15,12,0.800000,2485.0,4840.8
3,120,200,600,807,210,102,4.434066,4.5,85,63,0.741176,870.0,1838.2
4,120,200,1200,205,99,57,1.126374,3.0,36,27,0.750000,1670.0,3776.2
5,120,200,1800,87,51,35,0.478022,2.0,23,17,0.739130,2511.0,4314.6
6,120,300,600,989,250,116,5.434066,5.0,97,73,0.752577,845.0,1723.0
7,120,300,1200,246,120,67,1.351648,2.0,44,29,0.659091,1641.5,3366.0
8,120,300,1800,100,63,41,0.549451,2.0,26,16,0.615385,2397.5,4346.1
9,300,100,600,992,255,121,5.450549,5.0,95,80,0.842105,904.0,1916.8


Baseline config:


,max_gap_s,distance_threshold_m,min_dwell_s,n_stays,files_with_stays,users_with_stays,mean_stays_per_user,median_stays_per_active_user,users_with_2plus_stays,repeat_location_users,repeat_location_user_rate,median_duration_s,p90_duration_s
13,300,200,1200,478,174,89,2.626374,4.0,69,52,0.753623,1621.5,3202.9


### Cách đọc kết quả 27 cấu hình

Mỗi dòng trong bảng là **một bộ tham số khác nhau** của stay-point detector.

Ta thay đổi 3 tham số:

```text
max_gap_s            → khoảng mất dữ liệu tối đa vẫn cho phép cùng sequence
distance_threshold_m → bán kính tối đa để xem các point thuộc cùng một stay
min_dwell_s          → thời gian tối thiểu để một candidate được công nhận là stay
```

Mục tiêu không phải tìm cấu hình tạo **nhiều stay nhất**, mà tìm một cấu hình cân bằng giữa:

```text
đủ stay để downstream sử dụng
+
đủ user có stay
+
có dấu hiệu quay lại cùng location
+
không quá dễ dãi với gap, radius hoặc dwell time
```

---

### Baseline được chọn

Cấu hình baseline là:

```text
max_gap_s            = 300 s   (5 phút)
distance_threshold_m = 200 m
min_dwell_s          = 1200 s  (20 phút)
```

Trên sample gồm 182 users và 851 trajectory files, cấu hình này cho:

| metric                           |                   value |
| -------------------------------- | ----------------------: |
| detected stays                   |                 **478** |
| files có ít nhất 1 stay          |                 **174** |
| users có ít nhất 1 stay          |                  **89** |
| trung bình stays / user          |               **2.626** |
| median stays / active user       |                   **4** |
| users có >=2 stays               |                  **69** |
| users quay lại location gần nhau |                  **52** |
| repeat-location rate             |              **75.36%** |
| median stay duration             | **1621.5 s (~27 phút)** |
| p90 stay duration                | **3202.9 s (~53 phút)** |

Hai metric dễ nhầm:

* `mean_stays_per_user = 2.626`: tính trên **toàn bộ 182 users**, kể cả user không có stay.
* `median_stays_per_active_user = 4`: chỉ tính trên những user thực sự có ít nhất một stay.

`repeat_location_user_rate = 75.36%` nghĩa là:

```text
69 users có ít nhất 2 stays
↓
52 users có ít nhất một cặp stays cách nhau <= 200 m
↓
52 / 69 = 75.36%
```

Đây chỉ là dấu hiệu user quay lại cùng một vùng, **không có nghĩa vùng đó chắc chắn là Home hoặc Office**.

---

### Ảnh hưởng của `max_gap_s`

Để xem riêng ảnh hưởng của gap, ta giữ:

```text
radius = 200 m
dwell  = 20 phút
```

và chỉ thay đổi `max_gap_s`.

|   max gap-- |   stays | users có stay | repeat-location rate |
| --------: | ------: | ------------: | -------------------: |
|     120 s |     205 |            57 |               75.00% |
| **300 s** | **478** |        **89** |           **75.36%** |
|     600 s |     655 |           113 |               77.01% |

Khi tăng gap từ 2 phút → 5 phút → 10 phút, nhiều đoạn trajectory được giữ liên tục hơn nên số stay và số user có stay tăng mạnh.

Tuy nhiên repeat-location rate chỉ thay đổi nhẹ:

```text
75.00% → 75.36% → 77.01%
```

`600 s` tạo nhiều stay hơn, nhưng cũng cho phép nối qua khoảng mất dữ liệu dài tới 10 phút.

Vì không có ground truth để chứng minh việc nối qua gap dài như vậy là đúng, ta chọn **300 s** làm mức trung gian thận trọng.

---

### Ảnh hưởng của `distance_threshold_m`

Giữ:

```text
gap   = 300 s
dwell = 20 phút
```

và thay đổi stay radius.

|    radius |   stays | users có stay | repeat-location rate |
| --------: | ------: | ------------: | -------------------: |
|     100 m |     324 |            77 |               69.81% |
| **200 m** | **478** |        **89** |           **75.36%** |
|     300 m |     547 |           102 |               70.89% |

`100 m` khá chặt nên phát hiện ít stay hơn.

`300 m` cho nhiều stay và nhiều user hơn, nhưng cũng dễ gom các GPS point ở vùng rộng thành cùng một stay.

`200 m` nằm giữa hai mức này và trong ba cấu hình lân cận còn có repeat-location rate cao nhất.

Vì vậy **200 m** là lựa chọn cân bằng hợp lý cho baseline.

---

### Ảnh hưởng của `min_dwell_s`

Giữ:

```text
gap    = 300 s
radius = 200 m
```

và thay đổi thời gian dừng tối thiểu.

|   min dwell |   stays | users có stay | repeat-location rate | median duration |
| ----------: | ------: | ------------: | -------------------: | --------------: |
|     10 phút |   1,239 |           139 |               79.13% |        ~16 phút |
| **20 phút** | **478** |        **89** |           **75.36%** |    **~27 phút** |
|     30 phút |     210 |            67 |               61.54% |        ~40 phút |

Với `10 phút`, detector chấp nhận nhiều điểm dừng ngắn hơn nên số stay tăng rất mạnh.

Với `30 phút`, điều kiện quá chặt khiến nhiều stay và nhiều user bị loại.

`20 phút` nằm giữa hai phía và phù hợp với mục tiêu CP1 là tìm các **sustained stays** thay vì mọi điểm dừng ngắn.

---

### Kết luận

Baseline được giữ ở:

```text
300 s gap / 200 m radius / 20 phút dwell
```

không phải vì nó đạt giá trị lớn nhất ở mọi metric.

Ta chọn nó vì đây là một cấu hình **trung gian, dễ giải thích và tương đối ổn định** khi thay đổi từng tham số xung quanh baseline.

Quan trọng:

> Sensitivity analysis ở đây giúp chọn một engineering baseline hợp lý. Nó không chứng minh đây là bộ tham số tối ưu về độ chính xác, vì GeoLife không có ground truth Home/Office để đánh giá trực tiếp ở bước này.



## 4. Kết luận sensitivity và freeze CP1

### Điều surface cho thấy

- tăng `max_gap_s` → thường tăng coverage vì nhiều observations được giữ trong cùng continuity sequence;
- tăng stay radius → thường tăng khả năng candidate tồn tại lâu trong spatial neighborhood;
- tăng minimum dwell → giảm stay count và tăng duration của những stay còn lại;
- baseline không nằm ở một discontinuous edge của grid.

### Frozen CP1 engineering baseline

```text
same-second safe-collapse     10 m
continuity gap               300 s
hard-speed guard             1200 km/h
stay distance                200 m
minimum dwell                1200 s / 20 min
```

### Điều notebook này không chứng minh

- không chứng minh `200 m / 20 min` là accuracy-optimal;
- không chứng minh repeated-location proxy là HOME/OFFICE ground truth;
- không chứng minh transport-speed distributions nên được dùng làm cleaning thresholds.

### Follow-up mentor audit

Notebook 02c sau đó kiểm tra concern về same-second fast transportation. Kết luận: `10 m` là **safe-to-collapse threshold**, không phải corruption threshold. Groups >10 m vẫn tạo continuity boundary vì within-second ordering không recover được, nhưng diagnostic reason đổi thành `same_second_spatial_ambiguity`.

Boundary placement không đổi, vì vậy **không cần rerun** sensitivity surface này sau semantic amendment đó.